In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OrdinalEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    classification_report,
    confusion_matrix
)


# ============================================================
# 【读取步骤】
# 读取清洗后的训练集和测试集
# ============================================================

model_train = pd.read_parquet(
    "data/processed/model_train_clean.parquet"
)

model_test = pd.read_parquet(
    "data/processed/model_test_clean.parquet"
)


# ============================================================
# 【准备步骤】
# 分离特征、目标变量和客户编号
# ============================================================

# 1. 保存测试集客户编号
test_customer_ids = (
    model_test["SK_ID_CURR"]
    .copy()
)


# 2. 提取目标变量
y = (
    model_train["TARGET"]
    .astype("int8")
    .copy()
)


# 3. 建立训练特征
X = (
    model_train
    .drop(
        columns=[
            "TARGET",
            "SK_ID_CURR"
        ]
    )
    .copy()
)


# 4. 建立测试特征
X_test = (
    model_test
    .drop(
        columns=["SK_ID_CURR"]
    )
    .copy()
)


# 5. 划分训练集和验证集
X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)


print(
    "X_train Shape:",
    X_train.shape
)

print(
    "X_valid Shape:",
    X_valid.shape
)

print(
    "X_test Shape:",
    X_test.shape
)

print(
    "训练集违约率:",
    y_train.mean()
)

print(
    "验证集违约率:",
    y_valid.mean()
)

X_train Shape: (246008, 439)
X_valid Shape: (61503, 439)
X_test Shape: (48744, 439)
训练集违约率: 0.08072908198107379
验证集违约率: 0.08072776937710356


In [2]:
# ============================================================
# 【检查步骤】
# 区分类别特征和数值特征
# ============================================================

categorical_columns = (
    X_train
    .select_dtypes(
        include=[
            "object",
            "category"
        ]
    )
    .columns
    .tolist()
)


numeric_columns = (
    X_train
    .select_dtypes(
        include=["number"]
    )
    .columns
    .tolist()
)


print(
    "类别特征数量:",
    len(categorical_columns)
)

print(
    "数值特征数量:",
    len(numeric_columns)
)

print(
    "总特征数量:",
    len(categorical_columns)
    + len(numeric_columns)
)

print(
    "X_train 字段数量:",
    X_train.shape[1]
)

类别特征数量: 24
数值特征数量: 415
总特征数量: 439
X_train 字段数量: 439


In [3]:
# ============================================================
# 【数据预处理】
# 类别字段编码，数值字段保持原值
# ============================================================

tree_preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            "passthrough",
            numeric_columns
        ),
        (
            "categorical",
            OrdinalEncoder(
                handle_unknown="use_encoded_value",
                unknown_value=-1
            ),
            categorical_columns
        )
    ],
    remainder="drop"
)

In [8]:
# ============================================================
# 【树模型基线】
# 建立 Random Forest Pipeline
# ============================================================

random_forest_pipeline = Pipeline(
    steps=[
        (
            "preprocessor",
            tree_preprocessor
        ),
        (
            "model",
            RandomForestClassifier(
                n_estimators=150,
                max_depth=12,
                min_samples_split=30,
                min_samples_leaf=15,
                max_features="sqrt",
                class_weight="balanced_subsample",
                random_state=42,
                n_jobs=-1,
                verbose=0
            )
        )
    ]
)

In [9]:
# ============================================================
# 【模型训练】
# 训练 Random Forest
# ============================================================

random_forest_pipeline.fit(
    X_train,
    y_train
)

print(
    "Random Forest 训练完成"
)

Random Forest 训练完成


In [10]:
# ============================================================
# 【模型预测】
# 生成验证集违约概率
# ============================================================

rf_valid_probabilities = (
    random_forest_pipeline
    .predict_proba(X_valid)[:, 1]
)


# 暂时使用0.5阈值
rf_valid_predictions = (
    rf_valid_probabilities >= 0.5
).astype("int8")


print(
    "预测概率最小值:",
    rf_valid_probabilities.min()
)

print(
    "预测概率最大值:",
    rf_valid_probabilities.max()
)

print(
    "预测为违约的客户比例:",
    rf_valid_predictions.mean()
)

预测概率最小值: 0.09608294929506901
预测概率最大值: 0.849638210075867
预测为违约的客户比例: 0.1876981610653139


In [11]:
# ============================================================
# 【模型评估】
# 计算 Random Forest 验证集指标
# ============================================================

rf_roc_auc = roc_auc_score(
    y_valid,
    rf_valid_probabilities
)

rf_pr_auc = average_precision_score(
    y_valid,
    rf_valid_probabilities
)


print(
    "Random Forest Validation ROC-AUC:",
    round(rf_roc_auc, 4)
)

print(
    "Random Forest Validation PR-AUC:",
    round(rf_pr_auc, 4)
)


print(
    "\nClassification Report:"
)

print(
    classification_report(
        y_valid,
        rf_valid_predictions,
        digits=4
    )
)


print(
    "Confusion Matrix:"
)

print(
    confusion_matrix(
        y_valid,
        rf_valid_predictions
    )
)

Random Forest Validation ROC-AUC: 0.7609
Random Forest Validation PR-AUC: 0.2337

Classification Report:
              precision    recall  f1-score   support

           0     0.9501    0.8395    0.8914     56538
           1     0.2140    0.4975    0.2992      4965

    accuracy                         0.8119     61503
   macro avg     0.5820    0.6685    0.5953     61503
weighted avg     0.8906    0.8119    0.8436     61503

Confusion Matrix:
[[47464  9074]
 [ 2495  2470]]


In [12]:
# ============================================================
# 【模型对比】
# 对比 Logistic Regression 与 Random Forest
# ============================================================

model_comparison = pd.DataFrame(
    {
        "model": [
            "Logistic Regression",
            "Random Forest"
        ],
        "roc_auc": [
            0.7803,
            rf_roc_auc
        ],
        "pr_auc": [
            0.2701,
            rf_pr_auc
        ]
    }
)


model_comparison.sort_values(
    "roc_auc",
    ascending=False
).round(4)

,model,roc_auc,pr_auc
0,Logistic Regression,0.7803,0.2701
1,Random Forest,0.7609,0.2337
